# Train Mean Pool Title

> **Note:** Outputs below are from a previous run. They haven't been regenerated with the updated `cold` feature and unified eval. Run this notebook from scratch for current results.

In [1]:
# ! pip install "pandas<=2.3.2" "numpy" "torch<=2.5" "matplotlib" "seaborn" "matplotlib-venn" "datasets" "ipykernel" "recbole" "kmeans-pytorch" "sentence-transformers"

In [2]:
import numpy as np

# For NumPy 2.0 compatibility with RecBole 1.2
np.float_ = np.float64
np.int_ = np.int64
np.complex_ = np.complex128
np.unicode_ = np.str_

# Ensure logging on notebook works even on Colab
import logging
logging.getLogger().handlers.clear()

In [3]:
from typing import Any
import torch
import torch.nn as nn
import pandas as pd
from recbole.config import Config
from recbole.data.dataloader import FullSortEvalDataLoader, AbstractDataLoader
from recbole.data import create_dataset, data_preparation
from recbole.model.abstract_recommender import GeneralRecommender
from recbole.trainer import Trainer
from recbole.utils import init_seed, init_logger, InputType, ModelType
from sentence_transformers import SentenceTransformer

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# --- Config ---
# Assume we have `*.train.inter`, `*.valid.inter`, `*.test.inter`
DATASET_NAME: str = "beauty" 
DATA_DIR: str = "../data"
SEED = 67
DEVICE = "mps" # Other options: "cpu", "cuda"

## Create dataset

In [5]:
config_dict: dict[str, Any] = {
    "data_path": DATA_DIR,
    "dataset": DATASET_NAME,
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "benchmark_filename": ["train", "valid", "test"],
    "load_col": {
        "inter": ["user_id", "item_id"],
        "user": ["user_id", "cold"],
        "item": ["item_id", "cold", "title"],
    },
    "epochs": 1,
    "train_batch_size": 1024,
    "eval_batch_size": 409_600_000,
    # Must not use negative sampling so that as calculate_loss expects every interaction to be positive
    "train_neg_sample_args": None,
    "eval_args": {
        # Split is already determined by the `benchmark filename` as separate `.inter` files
        "split": None, 
        "order": "TO",
        "mode": {"valid": "full", "test": "full"},
    },
    "metrics": ["NDCG", "Recall", "MRR"],
    "topk": [20],
    "valid_metric": "NDCG@20",
    "seed": SEED,
}

config: Config = Config(model="Pop", config_dict=config_dict)
config.final_config_dict["device"] = torch.device(DEVICE)

init_logger(config)
init_seed(SEED, reproducibility=True)

In [6]:
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/data/dataset/dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
21 Jun 13:33    INFO  [Training]: train_batch_size = [1024] train_neg_sample_args: [{'distribution': 'none', 'sample_num': 'none', 'alpha': 'none', 'dynamic': False, 'candidate_num': 0}]
21 Jun 13:33    INFO  [Evaluation]: eval_batch_size = [409600000] eval_args: [{'split': None, 'order': 'TO', 'group_by': 'user', 'mode': {'va

## Define MeanPoolRecommender

In [7]:
class MeanPoolRecommender(GeneralRecommender):
    """Mean-pooling recommender.

    Encodes item titles with BGE text embeddings, then represents each user
    as the mean of their training item BGE embeddings. Scores are dot products
    between the user representation and item BGE embeddings.

    No learned parameters — pure mean-pooling baseline.
    """
    input_type = InputType.POINTWISE
    type = ModelType.TRADITIONAL

    def __init__(self, config, dataset):
        super().__init__(config, dataset)

        # ── Step 1: Get title strings via RecBole's data pipeline ──
        title_tokens: torch.Tensor = dataset.item_feat["title"]  # (n_items,) token IDs
        id2token: dict[int, str] = {
            v: k for k, v in dataset.field2token_id["title"].items()
        }
        titles: list[str] = []
        for tok in title_tokens:
            t = id2token.get(tok.item(), "")
            titles.append(t)

        # ── Step 2: Encode with BGE text encoder ──
        self.logger.info(
            "Encoding %d item titles with BGE (BAAI/bge-base-en-v1.5)...", self.n_items
        )
        bge_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device=config["device"])
        embs = bge_model.encode(titles, show_progress_bar=True, batch_size=256, normalize_embeddings=True)
        embs = torch.from_numpy(embs).float()  # (n_items, 768)

        # Zero out padding item (index 0) so it never gets recommended
        with torch.no_grad():
            embs[0] = 0.0

        self.register_buffer("item_embeddings", embs)
        self.register_buffer(
            "user_embed_sum", torch.zeros(self.n_users, 768)
        )
        self.register_buffer(
            "user_embed_cnt", torch.zeros(self.n_users, 1, dtype=torch.float)
        )

        # Fake parameter so Trainer's optimizer has something to reference
        self.fake_loss = nn.Parameter(torch.zeros(1))

    def calculate_loss(self, interaction):
        """Accumulate item embeddings per user from training batches."""
        user = interaction[self.USER_ID]
        item = interaction[self.ITEM_ID]
        emb = self.item_embeddings[item]

        self.user_embed_sum.index_add_(0, user, emb)
        self.user_embed_cnt.index_add_(
            0, user, torch.ones_like(user, dtype=torch.float).unsqueeze(-1)
        )
        return self.fake_loss

    def predict(self, interaction):
        """Score for (user, item) pairs — used in uni100 eval."""
        user = interaction[self.USER_ID]
        item = interaction[self.ITEM_ID]
        cnt = self.user_embed_cnt[user].float().clamp(min=1)
        user_emb = self.user_embed_sum[user] / cnt
        item_emb = self.item_embeddings[item]
        return (user_emb * item_emb).sum(dim=-1)

    def full_sort_predict(self, interaction):
        """Score for user × ALL items — used in full eval."""
        user = interaction[self.USER_ID]
        cnt = self.user_embed_cnt[user].float().clamp(min=1)
        user_emb = self.user_embed_sum[user] / cnt
        return torch.matmul(user_emb, self.item_embeddings.t()).view(-1)

## Train MeanPool

In [8]:
# Suppress httpx INFO logs from SentenceTransformer's CLIP model loading
logging.getLogger('httpx').setLevel(logging.WARNING)

model: MeanPoolRecommender = MeanPoolRecommender(config, train_data.dataset).to(config["device"])
trainer: Trainer = Trainer(config, model)

best_valid_score, best_valid_result = trainer.fit(train_data, valid_data)

21 Jun 13:33    INFO  Encoding 250853 item titles with BGE (BAAI/bge-base-en-v1.5)...
21 Jun 13:33    INFO  Loading SentenceTransformer model from BAAI/bge-base-en-v1.5.
Batches: 100%|██████████| 980/980 [06:48<00:00,  2.40it/s]
/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:235: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler(enabled=self.enable_scaler)
21 Jun 13:40    INFO  epoch 0 training [time: 10.17s, train loss: -603.3470]
21 Jun 13:41    INFO  epoch 0 evaluating [time: 19.90s, valid_score: 0.003500]
21 Jun 13:41    INFO  valid result: 
ndcg@20 : 0.0035    recall@20 : 0.0068    mrr@20 : 0.0036
21 Jun 13:41    INFO  Saving current: saved/Pop-Jun-21-2026_13-40-42.pth


In [9]:
print(f"\nBest valid score: {best_valid_score:.4f}")
print("Best valid result:")
for metric, score in best_valid_result.items():
    print(f"  {metric}: {score:.4f}")


Best valid score: 0.0035
Best valid result:
  ndcg@20: 0.0035
  recall@20: 0.0068
  mrr@20: 0.0036


## Evaluate on test set

In [10]:
test_result: dict[str, float] = trainer.evaluate(test_data, load_best_model=False)

print("Test results (Overall):")
for metric, value in test_result.items():
    print(f"  {metric}: {value:.4f}")

/Users/nginyc/repos/amazon-item-recommender/.venv/lib/python3.12/site-packages/recbole/trainer/trainer.py:583: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = tor

Test results (Overall):
  ndcg@20: 0.0031
  recall@20: 0.0057
  mrr@20: 0.0034


In [ ]:
def evaluate_on_subset(
    data: AbstractDataLoader,
    mask: np.ndarray,
    label: str
):
    inter_feat = data.dataset.inter_feat
    cat_ds = data.dataset.copy(inter_feat[mask])
    cat_dl = FullSortEvalDataLoader(config, cat_ds, sampler=data._sampler)
    results = trainer.evaluate(cat_dl)
    rows.append({
        "Segment": label,
        "Interactions": int(mask.sum()),
        **results,
    })

rows = []

cold_to_label = {0.0: "warm", 1.0: "cold"}

def evaluate_by_column(entity_feat, id_field, inter_id_array, entity_name):
    id_to_cold = dict(zip(
        entity_feat[id_field].numpy(),
        entity_feat["cold"].numpy(),
    ))
    for cold_val, label in cold_to_label.items():
        ids = {eid for eid, c in id_to_cold.items() if c == cold_val}
        mask = np.isin(inter_id_array, list(ids))
        if not mask.any():
            print(f"  {entity_name}-{label}: no interactions — skipping")
            continue
        evaluate_on_subset(test_data, mask, f"{entity_name}-{label}")

# Evaluation by user segments
evaluate_by_column(
    dataset.user_feat, dataset.uid_field,
    test_data.dataset.inter_feat[dataset.uid_field].numpy(),
    "user"
)

# Evaluation by item segments
evaluate_by_column(
    dataset.item_feat, dataset.iid_field,
    test_data.dataset.inter_feat[dataset.iid_field].numpy(),
    "item"
)

# Cross-tabulation: user × item segments
uid_to_cold = dict(zip(
    dataset.user_feat[dataset.uid_field].numpy(),
    dataset.user_feat["cold"].numpy(),
))
iid_to_cold = dict(zip(
    dataset.item_feat[dataset.iid_field].numpy(),
    dataset.item_feat["cold"].numpy(),
))

uid_array = test_data.dataset.inter_feat[dataset.uid_field].numpy()
iid_array = test_data.dataset.inter_feat[dataset.iid_field].numpy()

for uc_val, uc_label in cold_to_label.items():
    for ic_val, ic_label in cold_to_label.items():
        uc_uids = {uid for uid, c in uid_to_cold.items() if c == uc_val}
        ic_iids = {iid for iid, c in iid_to_cold.items() if c == ic_val}
        mask = np.isin(uid_array, list(uc_uids)) & np.isin(iid_array, list(ic_iids))
        if not mask.any():
            print(f"  user-{uc_label}×item-{ic_label}: no interactions — skipping")
            continue
        evaluate_on_subset(test_data, mask, f"user-{uc_label}×item-{ic_label}")

# Display results sorted by NDCG
df_results = pd.DataFrame(rows).sort_values("ndcg@20", ascending=False)
display(df_results)


21 Jun 13:41    INFO  Loading model structure and parameters from saved/Pop-Jun-21-2026_13-40-42.pth



Evaluation (warm)
--------------------
  Interactions: 46742
  ndcg@20: 0.0009
  recall@20: 0.0017
  mrr@20: 0.0011


21 Jun 13:41    INFO  Loading model structure and parameters from saved/Pop-Jun-21-2026_13-40-42.pth



Evaluation (cold)
--------------------
  Interactions: 150964
  ndcg@20: 0.0034
  recall@20: 0.0063
  mrr@20: 0.0038


: 